# mid_t Sensitivity Sweep — Lite

Single-checkpoint mid_t sweep for the COMP447 progress report.

- **Checkpoint**: 1980 kimg ECT (`network-snapshot-000198.pkl`)
- **mid_t values**: 8 points across [0.1, 2.5]
- **Samples per point**: 5000 (5k FID screening)
- **Expected runtime**: 15-25 minutes on Colab G4
- **Output**: `project/results/midt_sweep/lite_results.csv` + `lite_curve.png`

**How to use**: Connect this notebook to the Colab G4 runtime via the VS Code Colab extension, then run cells top-to-bottom. The kernel runs remotely; outputs and saved files come back into the repo via the synced workspace.

## Cell 1: Verify GPU and pull latest scripts

Confirms the Colab session has a CUDA GPU and pulls the latest `midt_sweep_lite.py` from GitHub.

In [ ]:
import os, subprocess, torch
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")

REPO = "/content/COMP447"
if not os.path.exists(REPO):
    print("Cloning repo...")
    subprocess.check_call(["git", "clone", "https://github.com/bakaraman/COMP447.git", REPO])
else:
    subprocess.check_call(["git", "-C", REPO, "fetch", "origin", "main"])
    subprocess.check_call(["git", "-C", REPO, "reset", "--hard", "origin/main"])
print("Repo synced to:", REPO)

## Cell 2: Locate the 1980 kimg checkpoint

Tries the default repo path first, then falls back to common Drive locations. If none of these work, edit `CHECKPOINT` manually and re-run.

In [ ]:
CANDIDATES = [
    "/content/COMP447/project/results_backup/ect_checkpoints/network-snapshot-000198.pkl",
    "/content/drive/MyDrive/COMP447/ect_checkpoints/network-snapshot-000198.pkl",
    "/content/drive/MyDrive/COMP447/network-snapshot-000198.pkl",
    "/content/network-snapshot-000198.pkl",
]
CHECKPOINT = None
for path in CANDIDATES:
    if os.path.exists(path):
        CHECKPOINT = path
        break

if CHECKPOINT is None:
    raise FileNotFoundError(
        "Could not find the 1980 kimg snapshot at any expected path.\n"
        "Mount Drive and edit CANDIDATES, or set CHECKPOINT directly.\n"
        f"Tried:\n  - " + "\n  - ".join(CANDIDATES)
    )

print(f"Checkpoint: {CHECKPOINT}")
print(f"Size: {os.path.getsize(CHECKPOINT)/1e6:.1f} MB")

## Cell 3: Setup the ECT environment (one-time)

ECT brings its own pinned versions and a small monkey-patch for PyTorch 2.x. This cell calls the project's setup script idempotently — safe to re-run.

In [ ]:
setup_log = subprocess.run(
    ["bash", f"{REPO}/project/scripts/setup_ect.sh"],
    capture_output=True, text=True,
)
print(setup_log.stdout[-2000:])
if setup_log.returncode != 0:
    print("STDERR:", setup_log.stderr[-2000:])
    raise RuntimeError("setup_ect.sh failed")

## Cell 4: Run the lite sweep

8 mid_t values × 5000 samples × FID via EDM `fid.py`. Streams progress to the cell output. CSV is rewritten after each completed point so partial progress is recoverable if the kernel disconnects.

In [ ]:
import sys
cmd = [
    "python3", f"{REPO}/project/scripts/midt_sweep_lite.py",
    "--checkpoint", CHECKPOINT,
    "--num", "5000",
    "--gen_batch", "64",
    "--fid_batch", "64",
    "--output_dir", "project/results/midt_sweep",
    "--repo_root", REPO,
]
print("$", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
proc.wait()
print("\nreturn code:", proc.returncode)

## Cell 5: Inspect the CSV

Sanity check: 8 rows of (mid_t, FID, n_samples, wall_s).

In [ ]:
import pandas as pd
csv_path = f"{REPO}/project/results/midt_sweep/lite_results.csv"
df = pd.read_csv(csv_path)
df_sorted = df.sort_values("fid").reset_index(drop=True)
print("All results (sorted by FID):")
print(df_sorted.to_string(index=False))
print("\nBest mid_t:", float(df_sorted.iloc[0]['mid_t']), "FID:", float(df_sorted.iloc[0]['fid']))
print("Default 0.821 FID:", float(df[df['mid_t']==0.821]['fid'].iloc[0]) if (df['mid_t']==0.821).any() else "--")

## Cell 6: Show the plot inline

In [ ]:
from IPython.display import Image, display
plot_path = f"{REPO}/project/results/midt_sweep/lite_curve.png"
display(Image(plot_path))

## Cell 7: Sync results back to local repo (optional)

If you want the CSV and PNG to live alongside the local project directory, the easiest path is to download them via the Files panel in Colab, or push from the Colab side back to GitHub. The cell below pushes the new results from the Colab git checkout.

In [ ]:
import subprocess
rs = lambda cmd: subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(rs(["git", "add", "project/results/midt_sweep"]).stdout)
print(rs(["git", "-c", "user.email=colab@batuhan", "-c", "user.name=colab",
         "commit", "-m", "Add lite mid_t sweep results"]).stdout)
print("--- Now from your local terminal: git pull origin main ---")